<a href="https://colab.research.google.com/github/xiayicheng3-code/flyrank-ml-internship-starter-my-copy/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Given my lane concerning CTR / Engagement Opportunity Scoring, this is primarily a scoring task. Each content item receives a position- and volume-adjusted CTR opportunity score, which is then used to rank pages for human review. The decision is which visible, underperforming pages a reviewer should inspect first—not which revision is guaranteed to improve CTR. This is a supervised machine learning track

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from notebooks/ to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

My prediction target is 90-day CTR. For pages with valid position data and sufficient impressions, I will estimate expected CTR from average search position. The opportunity proxy is the positive gap between expected and observed CTR, adjusted for impression volume. This proxy identifies pages that under-capture clicks relative to their visibility; it does not estimate the causal benefit of modifying a page. days_since_last_update provides freshness context but is not a complete revision history or treatment label.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import mean_absolute_error

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Verify the starter dataset's grain and update information.
print("Rows:", len(df))
print(
    "Duplicate client-content rows:",
    df.duplicated(["client_id", "content_id"]).sum()
)
print(
    "Update/date columns:",
    [c for c in df.columns if "update" in c.lower() or "date" in c.lower()]
)

# Remove missing position values and low-volume, noisy observations.
eligible = df[
    (df["avg_position"] > 0) &
    (df["impressions_90d"] >= 100)
].copy()

# Position and CTR have a nonlinear relationship, so log-transform position
# while retaining an interpretable linear-regression baseline.
X = np.log1p(eligible[["avg_position"]])
y = eligible["ctr"]
groups = eligible["client_id"]

cv = GroupKFold(n_splits=5)
eligible["expected_ctr"] = cross_val_predict(
    LinearRegression(),
    X,
    y,
    groups=groups,
    cv=cv
)

# Positive gap = CTR below the position-based expectation.
eligible["ctr_gap"] = (
    eligible["expected_ctr"] - eligible["ctr"]
).clip(lower=0)

# Translate the gap into a volume-aware missed-click proxy.
eligible["opportunity_clicks_proxy"] = (
    eligible["ctr_gap"] / 100 * eligible["impressions_90d"]
)

mae = mean_absolute_error(y, eligible["expected_ctr"])

print(f"Eligible pages: {len(eligible):,}")
print(f"Grouped cross-validated MAE: {mae:.3f} CTR percentage points")

eligible[
    [
        "content_id",
        "avg_position",
        "impressions_90d",
        "ctr",
        "expected_ctr",
        "ctr_gap",
        "opportunity_clicks_proxy",
        "days_since_last_update",
    ]
].sort_values("opportunity_clicks_proxy", ascending=False).head(10)

Rows: 30000
Duplicate client-content rows: 0
Update/date columns: ['days_since_last_update']
Eligible pages: 22,006
Grouped cross-validated MAE: 0.230 CTR percentage points


,content_id,avg_position,impressions_90d,ctr,expected_ctr,ctr_gap,opportunity_clicks_proxy,days_since_last_update
26844,content_8c19996aa890,2.5,509252,0.15,0.461674,0.311674,1587.205439,20
6653,content_5fe46e04994d,4.2,517715,0.14,0.403697,0.263697,1365.199125,104
7678,content_8451fc6f034d,2.3,272144,0.03,0.470291,0.440291,1198.224875,20
3394,content_36ff89c8214e,7.3,295097,0.05,0.383712,0.333712,984.774565,104
29879,content_1a9e894be2e2,4.0,416180,0.23,0.466452,0.236452,984.066898,22
17812,content_aaef01a50def,5.4,517109,0.25,0.426151,0.176151,910.894230,22
26531,content_cb112fce36be,5.6,309910,0.16,0.421128,0.261128,809.260855,104
7445,content_c8e9d6ab9013,9.7,208678,0.00,0.342248,0.342248,714.195294,104
6903,content_c84a0ab98e90,7.8,223271,0.03,0.312201,0.282201,630.072113,20
26474,content_a7427266c305,5.7,201111,0.11,0.418673,0.308673,620.774762,104


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My primary success metric is persistent-underperformance precision at the available review capacity. The model succeeds when pages ranked near the top are more likely than lower-ranked pages or a simple baseline to remain below their position-adjusted expected CTR in a later, non-overlapping window. This evaluates prioritization for review; it does not claim that revising a selected page will cause its CTR to improve. The code compares the later shortfall rate in a fixed-size review queue with the overall base rate. This measures persistent underperformance prioritization, not whether editing a page would cause improvement.

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

MIN_IMPRESSIONS = 100
REVIEW_SHARE = 0.10  # Demonstration only; replace with actual review capacity

# Require usable observations in both non-overlapping click windows.
evaluation = df[
    (df["avg_position"] > 0) &
    (df["impressions_prev_30d"] >= MIN_IMPRESSIONS) &
    (df["impressions_last_30d"] >= MIN_IMPRESSIONS)
].copy()

# CTR is expressed in percentage points.
evaluation["ctr_prev_pp"] = (
    100 * evaluation["clicks_prev_30d"]
    / evaluation["impressions_prev_30d"]
)
evaluation["ctr_last_pp"] = (
    100 * evaluation["clicks_last_30d"]
    / evaluation["impressions_last_30d"]
)

# A transparent position adjustment.
position_bins = [0, 3, 5, 10, 20, 50, np.inf]
evaluation["position_band"] = pd.cut(
    evaluation["avg_position"],
    bins=position_bins,
    include_lowest=True
)

# Estimate the typical page-level CTR in each position band,
# separately for the earlier and later windows.
evaluation["expected_ctr_prev_pp"] = (
    evaluation.groupby("position_band", observed=True)["ctr_prev_pp"]
    .transform("median")
)
evaluation["expected_ctr_last_pp"] = (
    evaluation.groupby("position_band", observed=True)["ctr_last_pp"]
    .transform("median")
)

evaluation["shortfall_prev_pp"] = (
    evaluation["expected_ctr_prev_pp"] - evaluation["ctr_prev_pp"]
)
evaluation["shortfall_last_pp"] = (
    evaluation["expected_ctr_last_pp"] - evaluation["ctr_last_pp"]
)

# Earlier score: positive shortfall, weighted by observable volume.
evaluation["earlier_opportunity_score"] = (
    evaluation["shortfall_prev_pp"].clip(lower=0)
    * evaluation["impressions_prev_30d"] / 100
)

# Later observed proxy outcome.
evaluation["below_expected_later"] = (
    evaluation["shortfall_last_pp"] > 0
)

review_n = max(1, int(len(evaluation) * REVIEW_SHARE))
review_queue = evaluation.nlargest(
    review_n, "earlier_opportunity_score"
)

queue_precision = review_queue["below_expected_later"].mean()
eligible_base_rate = evaluation["below_expected_later"].mean()
lift = queue_precision / eligible_base_rate

print(f"Eligible pages: {len(evaluation):,}")
print(
    "Pages below their position-band expectation later:",
    f"{evaluation['below_expected_later'].sum():,}"
)
print(f"Eligible-page base rate: {eligible_base_rate:.1%}")
print(f"Top review-queue precision: {queue_precision:.1%}")
print(f"Lift over the eligible-page base rate: {lift:.2f}x")

Eligible pages: 15,615
Pages below their position-band expectation later: 7,617
Eligible-page base rate: 48.8%
Top review-queue precision: 80.9%
Lift over the eligible-page base rate: 1.66x


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row represents one pseudonymized content page belonging to one client. It contains trailing 90-day performance statistics and previous/recent 30-day subwindows. After eligibility filtering, each row remains one page considered for review.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Slice relevant to this lane.
lane_df = df[
    (df["avg_position"] > 0) &
    (df["impressions_90d"] >= 100)
].copy()

duplicate_content = lane_df.duplicated("content_id").sum()
duplicate_client_content = lane_df.duplicated(
    ["client_id", "content_id"]
).sum()

print(f"All rows: {len(df):,}")
print(f"Eligible page rows: {len(lane_df):,}")
print(f"Unique pages: {lane_df['content_id'].nunique():,}")
print(f"Duplicate content IDs: {duplicate_content}")
print(f"Duplicate client-page pairs: {duplicate_client_content}")
print(f"Clients represented: {lane_df['client_id'].nunique():,}")

assert duplicate_content == 0
assert duplicate_client_content == 0

display(
    lane_df[
        [
            "content_id",
            "client_id",
            "avg_position",
            "impressions_90d",
            "clicks_90d",
            "ctr",
            "impressions_prev_30d",
            "impressions_last_30d",
        ]
    ].head()
)

All rows: 30,000
Eligible page rows: 22,006
Unique pages: 22,006
Duplicate content IDs: 0
Duplicate client-page pairs: 0
Clients represented: 30


,content_id,client_id,avg_position,impressions_90d,clicks_90d,ctr,impressions_prev_30d,impressions_last_30d
0,content_304f48230142,client_f369cb89fc,10.6,3803,29,0.76,987,578
1,content_a1fb4e703a9e,client_4e07408562,20.3,15320,7,0.05,5915,2501
2,content_9aa793d4d895,client_7f2253d7e2,36.5,12581,11,0.09,6089,2382
3,content_331d6c4de07b,client_19581e27de,6.2,11751,58,0.49,4206,3626
4,content_d99b7a2d90ca,client_3fdba35f04,44.0,19140,24,0.13,6452,4211


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule could review pages whose earlier CTR falls within the bottom 10% of a broad position band. However, this rule uses arbitrary cutoffs and treats pages within each band as equally comparable. It ignores impression volume and reliability, smooth differences in position, client and content characteristics, and interactions among these signals. ML is justified only if it produces higher later-window precision than this rule at the same review capacity.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this onebaseline = evaluation.copy()

baseline = evaluation.copy()

# Transparent fixed rule:
# flag pages in the bottom 10% of earlier CTR within their position band.
baseline["ctr_10th_percentile"] = (
    baseline.groupby("position_band", observed=True)["ctr_prev_pp"]
    .transform(lambda values: values.quantile(0.10))
)

baseline["fixed_rule_review"] = (
    baseline["ctr_prev_pp"] <= baseline["ctr_10th_percentile"]
)

baseline["reason_code"] = np.where(
    baseline["fixed_rule_review"],
    "bottom_10pct_ctr_in_position_band",
    "not_flagged"
)

# Evaluate against the later observed shortfall proxy from Part 3.
flagged = baseline[baseline["fixed_rule_review"]]
rule_precision = flagged["below_expected_later"].mean()
base_rate = baseline["below_expected_later"].mean()

print(f"Eligible pages: {len(baseline):,}")
print(f"Pages flagged by fixed rule: {len(flagged):,}")
print(f"Eligible-page base rate: {base_rate:.1%}")
print(f"Fixed-rule later precision: {rule_precision:.1%}")
print(f"Precision lift over base rate: {rule_precision / base_rate:.2f}x")

# Show one weakness: the percentile rule ignores impression volume.
k = len(flagged)
high_volume_candidates = baseline.nlargest(
    k, "earlier_opportunity_score"
)
missed = high_volume_candidates[
    ~high_volume_candidates["fixed_rule_review"]
]

# Summarize the fixed rule without repeating earlier cells.
selected_n = int(baseline["fixed_rule_review"].sum())
selected_share = selected_n / len(baseline)

fixed_rule_pages = baseline[baseline["fixed_rule_review"]]
fixed_precision = fixed_rule_pages["below_expected_later"].mean()
base_rate = baseline["below_expected_later"].mean()

# Compare both rankings using exactly the same review capacity.
volume_ranked_same_k = baseline.nlargest(
    selected_n,
    "earlier_opportunity_score"
)
volume_precision = volume_ranked_same_k[
    "below_expected_later"
].mean()

# Pages chosen by the volume-adjusted ranking but not by the fixed rule.
ranking_disagreements = volume_ranked_same_k[
    ~volume_ranked_same_k["fixed_rule_review"]
]

# Count pages tied exactly at the percentile cutoff.
tied_at_cutoff = fixed_rule_pages[
    "ctr_prev_pp"
].eq(
    fixed_rule_pages["ctr_10th_percentile"]
).sum()

absolute_improvement = 100 * (fixed_precision - base_rate)

print("Fixed-rule baseline summary")
print(
    f"1. The nominal bottom-10% rule selected {selected_n:,} pages "
    f"({selected_share:.1%} of eligible pages)."
)
print(
    f"   It selected more than 10% partly because {tied_at_cutoff:,} "
    "selected pages were tied exactly at the percentile cutoff."
)
print(
    f"2. Of the pages selected by the fixed rule, {fixed_precision:.1%} "
    "remained below their expected CTR in the later window."
)
print(
    f"   This is {absolute_improvement:.1f} percentage points above "
    f"the {base_rate:.1%} eligible-page baseline."
)
print(
    f"3. At the same review capacity, the volume-adjusted ranking "
    f"achieved {volume_precision:.1%} later-window precision."
)
print(
    f"4. The volume-adjusted ranking selected "
    f"{len(ranking_disagreements):,} pages that the fixed rule did not select."
)
print(
    "   These are disagreements between two rankings, "
    "not confirmed mistakes or guaranteed editing opportunities."
)

Eligible pages: 15,615
Pages flagged by fixed rule: 4,997
Eligible-page base rate: 48.8%
Fixed-rule later precision: 65.8%
Precision lift over base rate: 1.35x
Fixed-rule baseline summary
1. The nominal bottom-10% rule selected 4,997 pages (32.0% of eligible pages).
   It selected more than 10% partly because 4,997 selected pages were tied exactly at the percentile cutoff.
2. Of the pages selected by the fixed rule, 65.8% remained below their expected CTR in the later window.
   This is 17.0 percentage points above the 48.8% eligible-page baseline.
3. At the same review capacity, the volume-adjusted ranking achieved 69.7% later-window precision.
4. The volume-adjusted ranking selected 2,499 pages that the fixed rule did not select.
   These are disagreements between two rankings, not confirmed mistakes or guaranteed editing opportunities.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.